In [1]:
%load_ext autoreload
%autoreload 2

%pip install -r requirements.txt
%pip install -e .
%pip install matplotlib

import selfmade.nb101 as nb101
import selfmade.nbcore as nbcore
from selfmade.evallog import EvolutionLogger

records = r'.\datas\nasbench_full.tfrecord'
space = nb101.NASBench101Space()
evaluator = nb101.NASBench101Evaluator(records, space)

Note: you may need to restart the kernel to use updated packages.
Obtaining file:///F:/vinh/vscs/fi
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Attempting uninstall: nasbench
    Found existing installation: nasbench 1.0
    Uninstalling nasbench-1.0:
      Successfully uninstalled nasbench-1.0
  Running setup.py develop for nasbench
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



Loading dataset from file... This may take a few minutes...
Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`
Loaded dataset in 115 seconds


In [2]:
from sklearn.ensemble import RandomForestRegressor
import secrets
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
# Full vanilla (100 evals -> 10 pop, 10 iters)
def vanilla_run(loops, pop):
    pop.evolve_loop(total_gen=loops)
    return pop

In [4]:
# Half vanilla, half guided (100 evals -> 10 pop, 5 val, 5 gui)
def guided_run(loops, pop):
    split = int(loops/2)
    print("Half vanilla")
    pop.evolve_loop(total_gen=split)

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1)
    fi = nbcore.FeatureImportance(space)
    imp = fi.pipeline(pop.data, rf)

    gui = nbcore.Guidance(space, imp, pop.config)
    gui.sortNsplit()
    gui.calculate_guidance()

    print("Half guided")
    pop.evolve_loop(total_gen=split, guidance = gui)
    return pop

In [5]:
def check_population(pop, label=""):
    expected_bits = (
        pop.space.layer_limit
    ) * (
        pop.space.layer_limit - 1
    ) // 2

    print(f"\n{label}")
    print("Population size:", len(pop.members))

    for i, genome in enumerate(pop.members):
        flat = genome.code.replace("-", "")

        assert len(flat) == expected_bits, (
            f"Genome {i}: expected {expected_bits} bits, "
            f"got {len(flat)}: {genome.code}"
        )

        assert all(bit in "01" for bit in flat), (
            f"Genome {i} contains invalid bits: {genome.code}"
        )

    print("All genomes have", expected_bits, "bits.")


In [11]:
logger = EvolutionLogger()
pop = nbcore.Population(
    population_size=10,  # small while debugging
    search_space=space,
    evaluator=evaluator,
    logger=logger,
)

pop.initialize()
pop.evolve_loop(total_gen=3)

logger.save("evolution_trace.json")

Best performance: 0.9363982081413269
Recording generation 0
Finished recording
Generation 1 - Best performance: 0.9390023946762085
Recording generation 1
Finished recording
Generation 2 - Best performance: 0.9406049847602844
Recording generation 2
Finished recording
Generation 3 - Best performance: 0.9406049847602844
Recording generation 3
Finished recording
